# Naive Bayes

In [21]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt

In [22]:

df = pd.read_csv("Covid Dataset.csv")
df1 = df.dropna()
df1.head()

,Breathing Problem,Fever,Dry Cough,Sore throat,Running Nose,Asthma,Chronic Lung Disease,Headache,Heart Disease,Diabetes,...,Fatigue,Gastrointestinal,Abroad travel,Contact with COVID Patient,Attended Large Gathering,Visited Public Exposed Places,Family working in Public Exposed Places,Wearing Masks,Sanitization from Market,COVID-19
0,Yes,Yes,Yes,Yes,Yes,No,No,No,No,Yes,...,Yes,Yes,No,Yes,No,Yes,Yes,No,No,Yes
1,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,No,No,...,Yes,No,No,No,Yes,Yes,No,No,No,Yes
2,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,...,Yes,Yes,Yes,No,No,No,No,No,No,Yes
3,Yes,Yes,Yes,No,No,Yes,No,No,Yes,Yes,...,No,No,Yes,No,Yes,Yes,No,No,No,Yes
4,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,...,No,Yes,No,Yes,No,Yes,No,No,No,Yes


In [23]:

for i in df1.columns:
    df1[i] = df1[i].apply(lambda x: 1 if x == "Yes" else 0)

df1.head()

,Breathing Problem,Fever,Dry Cough,Sore throat,Running Nose,Asthma,Chronic Lung Disease,Headache,Heart Disease,Diabetes,...,Fatigue,Gastrointestinal,Abroad travel,Contact with COVID Patient,Attended Large Gathering,Visited Public Exposed Places,Family working in Public Exposed Places,Wearing Masks,Sanitization from Market,COVID-19
0,1,1,1,1,1,0,0,0,0,1,...,1,1,0,1,0,1,1,0,0,1
1,1,1,1,1,0,1,1,1,0,0,...,1,0,0,0,1,1,0,0,0,1
2,1,1,1,1,1,1,1,1,0,1,...,1,1,1,0,0,0,0,0,0,1
3,1,1,1,0,0,1,0,0,1,1,...,0,0,1,0,1,1,0,0,0,1
4,1,1,1,1,1,0,1,1,1,1,...,0,1,0,1,0,1,0,0,0,1


In [24]:

Y = df1["COVID-19"]
X = df1.drop("COVID-19", axis=1) 

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [26]:
model1 = BernoulliNB() 
model2 = DecisionTreeClassifier() 
model3 = LogisticRegression(C=0.9973, penalty='l2', solver='lbfgs', max_iter=1000)  
model4 = SVC(kernel='rbf', gamma=1, C=0.8, probability=True) 

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)
model4.fit(X_train, y_train)

SVC(C=0.8, gamma=1, probability=True)

In [27]:
prob1 = model1.predict_proba(X_test)[:, 1] 
prob2 = model2.predict_proba(X_test)[:, 1]
prob3 = model3.predict_proba(X_test)[:, 1]
prob4 = model4.predict_proba(X_test)[:, 1]

In [28]:
weights = np.array([0.0550, 0.1178, 0.2634, 0.5638])

weighted_probs = (
    weights[0] * prob1 +
    weights[1] * prob2 +
    weights[2] * prob3 +
    weights[3] * prob4
)

In [29]:
thresholds = np.arange(0, 1.01, 0.01)
best_threshold = 0
best_f1 = 0
f1_scores = []

for t in thresholds:

    final_predictions = (weighted_probs >= t).astype(int)
    
    f1 = f1_score(y_test, final_predictions)
    f1_scores.append(f1)
 
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
        
print(f"Best threshold: {best_threshold}")
print(f"Corresponding F1 Score: {best_f1:.4f}")

Best threshold: 0.31
Corresponding F1 Score: 0.9903


In [30]:
final_predictions = (weighted_probs >= best_threshold).astype(int)

In [31]:
accuracy = accuracy_score(y_test, final_predictions)
precision = precision_score(y_test, final_predictions)
recall = recall_score(y_test, final_predictions)

print(f"Best threshold: {best_threshold:.2f}") 
print(f"Accuracy: {accuracy:.4f}") 
print(f"Precision: {precision:.4f}") 
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {best_f1:.4f}") 

Best threshold: 0.31
Accuracy: 0.9844
Precision: 0.9943
Recall: 0.9864
F1 Score: 0.9903
